# 频域功率组间统计分析

对 ADHD 单纯组 (adhd)、ADHD 共患阅读困难组 (com)、正常发展组 (td) 进行频域功率的组间统计比较。

**分析内容:** 各频段功率 (Delta, Theta, Alpha, Beta, Gamma) — 共 5 项检验

**输出目录:** `reports/comparison/frequency/`

## 1. 导入和配置

In [15]:
import sys
from pathlib import Path

# 确保 stats_utils 可导入
sys.path.insert(0, str(Path('.').resolve()))
from stats_utils import *

setup_plotting()

# 输出路径
OUTPUT_PATH = BASE_PATH / 'reports' / 'comparison' / 'frequency'
OUTPUT_PATH, FIGURES_PATH = ensure_output_dirs(OUTPUT_PATH)

print(f'输出目录: {OUTPUT_PATH}')
print(f'图片目录: {FIGURES_PATH}')

输出目录: d:\LYW\REST_COM\reports\comparison\frequency
图片目录: d:\LYW\REST_COM\reports\comparison\frequency\figures


## 2. 数据加载

In [16]:
import pandas as pd

def load_frequency_data() -> pd.DataFrame:
    """加载三组频域数据 (subject_band_powers.csv)"""
    dfs = []
    for g in GROUPS:
        csv_path = FREQ_PATHS[g] / 'subject_band_powers.csv'
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            df['group'] = g
            dfs.append(df)
            print(f'  {GROUP_LABELS[g]}: {len(df)} 被试')
        else:
            print(f'  ⚠ 未找到: {csv_path}')
    if not dfs:
        return pd.DataFrame()
    return pd.concat(dfs, ignore_index=True)


print('--- 加载频域数据 ---')
freq_df = load_frequency_data()
print(f'\n频域数据: {freq_df.shape}')
freq_df.head()

--- 加载频域数据 ---
  ADHD单纯组: 12 被试
  ADHD共患阅读困难组: 14 被试
  正常发展组: 8 被试

频域数据: (34, 7)


,subject_id,group,Delta,Theta,Alpha,Beta,Gamma
0,18,adhd,2.813040e-11,3.262314e-12,1.681465e-12,2.464228e-13,6.749327e-14
1,29,adhd,1.199479e-11,5.302395e-12,1.799600e-12,3.183868e-13,8.900980e-14
2,31,adhd,1.109306e-11,1.293141e-12,4.601824e-13,1.975893e-13,1.647121e-13
3,37,adhd,2.466549e-11,1.037310e-11,4.341498e-12,1.448976e-12,8.100163e-13
4,38,adhd,9.824419e-12,3.154986e-12,2.931533e-12,4.089642e-13,2.397410e-13


## 3. Omnibus 检验

In [17]:
print('=' * 60)
print('频域指标 Omnibus 检验')
print('=' * 60)

freq_results = []
if not freq_df.empty:
    for band in FREQ_BANDS:
        if band not in freq_df.columns:
            continue
        res = run_analysis_for_metric(freq_df, band, 'band_power', freq_band=band)
        if res:
            freq_results.append(res)
            sig_mark = '***' if res['p'] < 0.001 else '**' if res['p'] < 0.01 else '*' if res['p'] < 0.05 else ''
            print(f"  {band:<8s} | {res['test']:<16s} | "
                  f"p={res['p']:.4f} {sig_mark:>3s} | {res['es_name']}={res['effect_size']:.3f}")

print(f'\n频域检验总数: {len(freq_results)}')

频域指标 Omnibus 检验
  Delta    | Kruskal-Wallis   | p=0.5463     | ε²=-0.026
  Theta    | Kruskal-Wallis   | p=0.5517     | ε²=-0.026
  Alpha    | Kruskal-Wallis   | p=0.8592     | ε²=-0.055
  Beta     | Kruskal-Wallis   | p=0.8739     | ε²=-0.056
  Gamma    | Kruskal-Wallis   | p=0.5812     | ε²=-0.030

频域检验总数: 5


## 4. FDR 校正

In [18]:
freq_omnibus_df = apply_fdr(freq_results)

if not freq_omnibus_df.empty:
    print('频域 Omnibus 结果 (FDR 校正后):')
    print(freq_omnibus_df[['metric', 'freq_band', 'test', 'p', 'p_fdr', 'significant', 'effect_size']].to_string(index=False))

# 汇总事后检验
posthoc_df = collect_posthoc(freq_results)

if not posthoc_df.empty:
    print('\n事后检验结果 (仅 omnibus 显著指标):')
    print(posthoc_df[['metric', 'freq_band', 'pair', 'test', 'p', 'cohens_d', 'significant']].to_string(index=False))

# 保存 CSV
freq_omnibus_df.to_csv(OUTPUT_PATH / 'frequency_omnibus_stats.csv', index=False, encoding='utf-8-sig')
if not posthoc_df.empty:
    posthoc_df.to_csv(OUTPUT_PATH / 'posthoc_stats.csv', index=False, encoding='utf-8-sig')
print('\n✓ 统计结果已保存')

频域 Omnibus 结果 (FDR 校正后):
    metric freq_band           test        p    p_fdr  significant  effect_size
band_power     Delta Kruskal-Wallis 0.546314 0.873931        False    -0.025512
band_power     Theta Kruskal-Wallis 0.551652 0.873931        False    -0.026139
band_power     Alpha Kruskal-Wallis 0.859185 0.873931        False    -0.054724
band_power      Beta Kruskal-Wallis 0.873931 0.873931        False    -0.055822
band_power     Gamma Kruskal-Wallis 0.581225 0.873931        False    -0.029509

✓ 统计结果已保存


## 5. 可视化

In [19]:
# 频域箱线图
if not freq_df.empty:
    for band in FREQ_BANDS:
        if band in freq_df.columns:
            plot_boxplots(freq_df, band,
                          f'{band} 频段功率', f'{band} Power (μV²/Hz)',
                          FIGURES_PATH / f'boxplot_freq_{band}.png')
    print('✓ 频域箱线图已保存')

✓ 频域箱线图已保存


In [20]:
# 效应量森林图
plot_effect_sizes(posthoc_df, FIGURES_PATH / 'effect_sizes_forest.png')

# 显著性热图
plot_significance_heatmap(freq_omnibus_df, '频域指标 Omnibus p 值 (FDR)',
                          FIGURES_PATH / 'heatmap_freq_significance.png')
print('✓ 可视化完成')

  无显著事后检验结果, 跳过森林图
✓ 可视化完成


## 6. 生成报告

In [21]:
# 描述性统计
desc_sections = ['### 频域指标\n']
if not freq_df.empty:
    for band in FREQ_BANDS:
        if band in freq_df.columns:
            desc_sections.append(f'**{band}:**\n')
            desc_sections.append(format_desc_table(freq_df, band))
desc_text = '\n'.join(desc_sections)

# Omnibus 表格
omnibus_text = format_omnibus_table(freq_omnibus_df, '频域指标')

# 事后检验表格
posthoc_text = format_posthoc_table(posthoc_df)

# 样本量
sample_sizes = {g: len(freq_df[freq_df['group'] == g]) for g in GROUPS}

report = generate_report(
    title='频域功率组间统计比较报告',
    analysis_desc='频域功率: Delta, Theta, Alpha, Beta, Gamma (5 检验)',
    sample_sizes=sample_sizes,
    desc_text=desc_text,
    omnibus_text=omnibus_text,
    posthoc_text=posthoc_text,
)

report_file = OUTPUT_PATH / 'frequency_comparison_report.md'
with open(report_file, 'w', encoding='utf-8') as f:
    f.write(report)
print(f'✓ 报告已保存: {report_file}')

✓ 报告已保存: d:\LYW\REST_COM\reports\comparison\frequency\frequency_comparison_report.md


## 7. 分析完成

In [22]:
print('=' * 60)
print('频域功率统计分析完成!')
print('=' * 60)

print(f'\n输出文件:')
for f in sorted(OUTPUT_PATH.glob('*')):
    if f.is_file():
        print(f'  - {f.name}')

print(f'\n可视化文件:')
for f in sorted(FIGURES_PATH.glob('*.png')):
    print(f'  - {f.name}')

# 显著结果摘要
print('\n' + '=' * 60)
print('显著结果摘要 (FDR < 0.05)')
print('=' * 60)

if not freq_omnibus_df.empty:
    sig = freq_omnibus_df[freq_omnibus_df['significant']]
    print(f'\n频域功率: {len(sig)}/{len(freq_omnibus_df)} 项显著')
    if len(sig) > 0:
        print(sig[['metric', 'freq_band', 'p_fdr', 'effect_size']].to_string(index=False))
    else:
        print('  无显著结果')

频域功率统计分析完成!

输出文件:
  - frequency_comparison_report.md
  - frequency_omnibus_stats.csv

可视化文件:
  - boxplot_freq_Alpha.png
  - boxplot_freq_Beta.png
  - boxplot_freq_Delta.png
  - boxplot_freq_Gamma.png
  - boxplot_freq_Theta.png
  - heatmap_freq_significance.png

显著结果摘要 (FDR < 0.05)

频域功率: 0/5 项显著
  无显著结果
